### Configure API Keys

In [ ]:
import os
from getpass import getpass

# Load environment variables from .env file if it exists
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# Check if the required API keys are set in the environment variables, 
# if not prompt the user to enter them
if not os.getenv('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass("Enter your GROQ API key: ")

if not os.getenv('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = getpass("Enter your Tavily API key: ")

if not os.getenv('PINECONE_API_KEY'):
    os.environ['PINECONE_API_KEY'] = getpass("Enter your Pinecone API key: ")

# Print the status of the API keys
print('GROQ_API_KEY configured:', bool(os.getenv('GROQ_API_KEY')))
print('TAVILY_API_KEY configured:', bool(os.getenv('TAVILY_API_KEY')))
print('PINECONE_API_KEY configured:', bool(os.getenv('PINECONE_API_KEY')))

### Load Real Docs

We will create our private knowledge base from a real web document:

https://docs.langchain.com/oss/python/langgraph/agentic-rag

This becomes our internal/private KB for the demo.

The web fallback is separate. That means:

-> First source: indexed LangGraph documentation page

-> Fallback source: live web search through Tavily

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

SOURCE_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

# Load documents from the specified URL using WebBaseLoader
loader = WebBaseLoader(
    web_path=(SOURCE_URL,),
    requests_kwargs={
        'headers': {
            'User-Agent': 'Mozilla/5.0 Agentic-RAG-Industry-Demo'
        }
    }
)

raw_docs = loader.load()

print('Loaded documents', len(raw_docs))
print('Source: ', raw_docs[0].metadata.get('source'))
print('\nPreview: \n')
print(raw_docs[0].page_content[:1500])

### Split Documents into Chunks

In [ ]:
from langchain_text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)

chunks = splitter.split_documents(raw_docs)

print('Total chunks created:', len(chunks))
print('Preview of first chunk:\n')
print(chunks[0].page_content[:900])

### Create Free Local Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

sample_vector = embeddings.embed_query("What is Agentic RAG?")
print('Embedding dimension:', len(sample_vector))

### Create the Pinecone Vector Database

In [ ]:
import time
from pinecone import Pinecone, SeverlessSpec
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

# Connect to Pinecone
pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

# Create the index only if it does not already exist
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,       #all MiniLM-L6-v2 embedding dimension
        metric="cosine",
        spec=SeverlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

    # wait until Pinecone reports the new index as ready
    while not pc.describe_index(INDEX_NAME).status['ready']:
        print("Waiting for index to be ready...")
        time.sleep(5)

print(f"Index '{INDEX_NAME}' is ready.")

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print("Pinecone vector database and retriever are ready.")